# Lab 6: LLM Architecture Anatomy

## What this notebook does

Implements `ModelConfig`, a dataclass holding a transformer's hyperparameters (`vocab_size`, `d_model`, `n_layers`, `n_heads`, `d_ff`), with methods that compute how many parameters each component -- embedding, attention, feed-forward, layer norms, output head -- contains. `count_parameters` adds them all up into a total, keyed by component.

The full explanation of parameter counting, dataclasses as config objects, and model config dicts lives in the reading pane above.

In [ ]:
from dataclasses import dataclass

### Concept 1 practice: counting things programmatically

Same rows-times-cols multiplication as the floor-tiles example, this time counting auditorium seats.

In [ ]:
rows = 20
seats_per_row = 15
total_seats = rows * seats_per_row
print(total_seats)  # 300


### Concept 2 practice: dataclasses as config objects

Same `@dataclass` bundle-of-named-values pattern as `Rectangle`, this time a car's driving range.

In [ ]:
from dataclasses import dataclass

@dataclass
class Car:
    mpg: float
    tank_gallons: float

    def range_miles(self) -> float:
        return self.mpg * self.tank_gallons

c = Car(mpg=30.0, tank_gallons=12.0)
print(c.range_miles())  # 360.0


### Concept 3 practice: config dicts as the shape of a real model

Same plain-dictionary-of-named-numbers idea as GPT-2's config, this time a small server config.

In [ ]:
server_config = {
    "host": "localhost",
    "port": 8080,
    "workers": 4,
    "timeout_seconds": 30,
}
print(server_config["workers"])  # 4


### Concept 4 practice: plotting with matplotlib

Same shifted-pair grouped bar chart as the computed-vs-published example, this time budget vs. actual spend.

In [ ]:
import matplotlib.pyplot as plt

quarters = ["Q1", "Q2"]
budget = [100, 120]
actual = [90, 130]
x = range(len(quarters))

plt.bar([i - 0.2 for i in x], budget, width=0.4, label="budget")
plt.bar([i + 0.2 for i in x], actual, width=0.4, label="actual")
plt.xticks(list(x), quarters)
plt.ylabel("$ thousands")
plt.legend()
plt.savefig("budget_vs_actual.png")
print("Saved grouped bar chart to budget_vs_actual.png")  # Saved grouped bar chart to budget_vs_actual.png


## Exercise

### Goal
Big language models are mostly big piles of numbers called **parameters**. Before training or running a model you can work out how many it has just from its settings, using multiplication. You will finish a small `ModelConfig` class that counts the parameters in each part of a simple transformer language model (the same kind of model behind chatbots), and the class adds them up for you.

### Words you will see
- **Parameter**: one learnable number inside the model. "7 billion parameters" means 7 billion such numbers.
- **Matrix of shape `a x b`**: a table of numbers with `a` rows and `b` columns. It holds `a * b` parameters.
- **Dataclass**: a Python class whose fields (`vocab_size`, `d_model`, ...) are listed at the top and stored for you. You read them as `self.d_model`, etc.
- **`vocab_size`**: how many different tokens (word pieces) the model knows.
- **`d_model`**: the width of the model, i.e. how many numbers represent one token inside it.
- **`d_ff`**: the width of the feed-forward layer's inner step (usually about 4 times `d_model`).
- **`n_layers`**: how many identical transformer **blocks** are stacked. Each block has its own attention, feed-forward and layer-norm parameters.
- **`n_heads`**: how many attention heads. It is stored in the config but does **not** change the count in this exercise.
- **Embedding table**: a `vocab_size x d_model` table that turns a token id into its vector. (Already written for you.)
- **Attention**: the part where tokens look at each other. It uses 4 square projection matrices (query Q, key K, value V and output), each `d_model x d_model`. We ignore bias vectors.
- **Feed-forward**: two linear layers per block: `d_model -> d_ff` and `d_ff -> d_model`.
- **Layer norm**: a small tidying step. Each one has a scale and a shift for every dimension, so `2 * d_model` parameters. Each block has 2 of them.
- **Output head**: the final `d_model -> vocab_size` matrix that scores every token. Assume it is a separate matrix (not shared with the embedding table).

### What you start with
The `ModelConfig` class already has `embedding_params` and `count_parameters` written. `count_parameters` calls the four methods you write and multiplies the per-block ones by `n_layers`. You only fill in four small methods; each takes no arguments besides `self` and returns an `int`.

Running example used below: `tiny = ModelConfig(vocab_size=1000, d_model=64, n_layers=2, n_heads=4, d_ff=256)`. Also a micro model `ModelConfig(vocab_size=10, d_model=4, n_layers=1, n_heads=2, d_ff=8)` for hand checks.

### `attention_params_per_layer(self)`
**Returns:** an `int`, the attention parameters in **one** block.

**Idea:** 4 matrices, each `d_model` by `d_model`, so 4 times `d_model` times `d_model`.

**Check:** micro model: `4 * 4 * 4 = 64`. For `tiny`: `4 * 64 * 64 = 16384`.

### `feed_forward_params_per_layer(self)`
**Returns:** an `int`, the feed-forward parameters in **one** block.

**Idea:** the first layer is a `d_model x d_ff` matrix, the second is `d_ff x d_model`. Add the two sizes (they are equal, so it is also twice one of them).

**Check:** micro model: `4*8 + 8*4 = 64`. For `tiny`: `64*256*2 = 32768`.

### `layernorm_params_per_layer(self)`
**Returns:** an `int`, the layer-norm parameters in **one** block.

**Idea:** 2 layer norms per block, each with `2 * d_model` numbers (a scale and a shift per dimension).

**Check:** micro model: `2 * 2 * 4 = 16`. For `tiny`: `2 * 2 * 64 = 256`.

### `output_head_params(self)`
**Returns:** an `int`, the size of the final `d_model x vocab_size` matrix. It does not depend on `n_layers`.

**Check:** micro model: `4 * 10 = 40`. For `tiny`: `64 * 1000 = 64000`.

### Putting it together (already coded)
For `tiny`, once your methods are right, `tiny.count_parameters()` returns
`{'embedding': 64000, 'attention': 32768, 'feed_forward': 65536, 'layernorm': 512, 'output_head': 64000, 'total': 226816}`.
The per-block numbers are doubled because `n_layers=2`. For the micro model the total is `40 + 64 + 64 + 16 + 40 = 224`.

### What the asserts test
The asserts build `tiny` and compare each of your four methods with the formula (`4*64*64`, `64*256*2`, `2*2*64`, `64*1000`). A failed assert prints the expected and the actual value. Seeing `All checks passed.` means all four counts are right, and the cell then prints the full parameter breakdown dictionary.

In [ ]:
@dataclass
class ModelConfig:
    vocab_size: int
    d_model: int
    n_layers: int
    n_heads: int
    d_ff: int

    def embedding_params(self) -> int:
        return self.vocab_size * self.d_model

    def attention_params_per_layer(self) -> int:
        """Parameters in the attention part of ONE block.

        Returns:
            int. Example: d_model=64 -> 16384.
        """
        # TODO:
        # 1. Attention has 4 projection matrices (Q, K, V, output).
        # 2. Each is d_model x d_model, i.e. d_model * d_model numbers.
        # 3. Return 4 times that. Ignore bias vectors.
        raise NotImplementedError("ModelConfig.attention_params_per_layer() is not implemented yet - replace the TODO with your code")

    def feed_forward_params_per_layer(self) -> int:
        """Parameters in the feed-forward part of ONE block.

        Returns:
            int. Example: d_model=64, d_ff=256 -> 32768.
        """
        # TODO:
        # 1. First layer is d_model x d_ff numbers.
        # 2. Second layer is d_ff x d_model numbers.
        # 3. Return the sum of the two sizes.
        raise NotImplementedError("ModelConfig.feed_forward_params_per_layer() is not implemented yet - replace the TODO with your code")

    def layernorm_params_per_layer(self) -> int:
        """Parameters in the layer norms of ONE block.

        Returns:
            int. Example: d_model=64 -> 256.
        """
        # TODO:
        # 1. One layer norm has 2 * d_model parameters (scale and shift).
        # 2. A block has 2 layer norms.
        # 3. Return the total for the block.
        raise NotImplementedError("ModelConfig.layernorm_params_per_layer() is not implemented yet - replace the TODO with your code")

    def output_head_params(self) -> int:
        """Parameters in the final output head (counted once, not per block).

        Returns:
            int. Example: d_model=64, vocab_size=1000 -> 64000.
        """
        # TODO:
        # 1. The head is a d_model x vocab_size matrix.
        # 2. It is separate from the embedding table (not shared), so count it in full.
        # 3. Return d_model * vocab_size.
        raise NotImplementedError("ModelConfig.output_head_params() is not implemented yet - replace the TODO with your code")

    def count_parameters(self) -> dict[str, int]:
        per_layer = (
            self.attention_params_per_layer()
            + self.feed_forward_params_per_layer()
            + self.layernorm_params_per_layer()
        )
        breakdown = {
            "embedding": self.embedding_params(),
            "attention": self.attention_params_per_layer() * self.n_layers,
            "feed_forward": self.feed_forward_params_per_layer() * self.n_layers,
            "layernorm": self.layernorm_params_per_layer() * self.n_layers,
            "output_head": self.output_head_params(),
        }
        breakdown["total"] = sum(v for k, v in breakdown.items() if k != "total")
        return breakdown


tiny = ModelConfig(vocab_size=1000, d_model=64, n_layers=2, n_heads=4, d_ff=256)
assert tiny.attention_params_per_layer() == 4 * 64 * 64, 'tiny.attention_params_per_layer() should be ' + repr(4 * 64 * 64) + ", got " + repr(tiny.attention_params_per_layer())
assert tiny.feed_forward_params_per_layer() == 64 * 256 * 2, 'tiny.feed_forward_params_per_layer() should be ' + repr(64 * 256 * 2) + ", got " + repr(tiny.feed_forward_params_per_layer())
assert tiny.layernorm_params_per_layer() == 2 * 2 * 64, 'tiny.layernorm_params_per_layer() should be ' + repr(2 * 2 * 64) + ", got " + repr(tiny.layernorm_params_per_layer())
assert tiny.output_head_params() == 64 * 1000, 'tiny.output_head_params() should be ' + repr(64 * 1000) + ", got " + repr(tiny.output_head_params())

print("All checks passed.")
print(tiny.count_parameters())

## Solution

In [ ]:
@dataclass
class ModelConfig:
    vocab_size: int
    d_model: int
    n_layers: int
    n_heads: int
    d_ff: int

    def embedding_params(self) -> int:
        return self.vocab_size * self.d_model

    def attention_params_per_layer(self) -> int:
        return 4 * self.d_model * self.d_model

    def feed_forward_params_per_layer(self) -> int:
        return self.d_model * self.d_ff + self.d_ff * self.d_model

    def layernorm_params_per_layer(self) -> int:
        return 2 * 2 * self.d_model

    def output_head_params(self) -> int:
        return self.d_model * self.vocab_size

    def count_parameters(self) -> dict[str, int]:
        per_layer = (
            self.attention_params_per_layer()
            + self.feed_forward_params_per_layer()
            + self.layernorm_params_per_layer()
        )
        breakdown = {
            "embedding": self.embedding_params(),
            "attention": self.attention_params_per_layer() * self.n_layers,
            "feed_forward": self.feed_forward_params_per_layer() * self.n_layers,
            "layernorm": self.layernorm_params_per_layer() * self.n_layers,
            "output_head": self.output_head_params(),
        }
        breakdown["total"] = sum(v for k, v in breakdown.items() if k != "total")
        return breakdown


print(ModelConfig(vocab_size=1000, d_model=64, n_layers=2, n_heads=4, d_ff=256).count_parameters())

## Real-world project

Computes parameter counts for real published GPT-2 configs and compares against their real published totals. GPT-2's output head *is* tied to the input embedding (they share one matrix), so we subtract that overlap to match the real numbers. Expect the computed total to land within about 1% of the published one, not exact - our formula skips positional embeddings and per-layer bias vectors to keep the exercise simple, and the leftover difference is almost entirely those two things.

In [ ]:
import matplotlib.pyplot as plt

%matplotlib inline

# Real, published GPT-2 configs (Radford et al., 2019) and their real total parameter counts.
REAL_CONFIGS = {
    "GPT-2 small": (ModelConfig(vocab_size=50257, d_model=768, n_layers=12, n_heads=12, d_ff=3072), 124_439_808),
    "GPT-2 medium": (ModelConfig(vocab_size=50257, d_model=1024, n_layers=24, n_heads=16, d_ff=4096), 354_823_168),
}

names, computed, published = [], [], []
for name, (config, real_total) in REAL_CONFIGS.items():
    breakdown = config.count_parameters()
    # GPT-2 ties the output head to the embedding table, so the real model doesn't
    # pay for both - subtract the tied duplicate to compare apples to apples.
    tied_total = breakdown["total"] - breakdown["output_head"]
    names.append(name)
    computed.append(tied_total)
    published.append(real_total)
    print(f"{name}: computed(tied)={tied_total:,}  published={real_total:,}  "
          f"diff={abs(tied_total - real_total):,}")
    print(f"  breakdown: {breakdown}")

x = range(len(names))
plt.figure(figsize=(7, 5))
plt.bar([i - 0.2 for i in x], computed, width=0.4, label="computed (tied)")
plt.bar([i + 0.2 for i in x], published, width=0.4, label="published")
plt.xticks(list(x), names)
plt.ylabel("parameters")
plt.title("Computed vs. published parameter counts")
plt.legend()
plt.tight_layout()
plt.show()